# Project 8: Student Degree Classification (Advanced - High Accuracy)

Build an advanced neural network to classify students into degree categories with high accuracy using improved techniques.

## Objectives
- Load and preprocess CSV data with advanced techniques
- Build a deep neural network with optimized architecture
- Implement class balancing for imbalanced data
- Use learning rate scheduling and early stopping
- Achieve high accuracy (>85%) through hyperparameter tuning
- Evaluate model with comprehensive metrics
- Analyze feature importance and model interpretability

## Improvements Over Project 7

1. **Deeper Architecture**: 7 → 64 → 128 → 64 → 32 → 5 (vs 7 → 32 → 16 → 5)
2. **Class Balancing**: SMOTE-like oversampling for minority classes
3. **Learning Rate Scheduling**: Adaptive learning rate with decay
4. **More Training**: 500 epochs with early stopping (vs 200)
5. **Better Initialization**: He initialization for ReLU layers
6. **Feature Engineering**: Polynomial features and interaction terms
7. **Cross-Validation**: K-fold validation for robust evaluation
8. **Ensemble Methods**: Multiple model averaging

## Degree Categories

Based on final score:
- **Bad**: score < 50
- **Acceptable**: 50 ≤ score < 65
- **Good**: 65 ≤ score < 75
- **Very Good**: 75 ≤ score < 85
- **Excellent**: score ≥ 85

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (classification_report, confusion_matrix, 
                           accuracy_score, f1_score, precision_score, recall_score)
from sklearn.utils.class_weight import compute_class_weight
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Import directly from module to avoid torch dependency
import importlib.util
from pathlib import Path
spec = importlib.util.spec_from_file_location(
    "neural_networks",
    Path('src') / "deep_learning" / "neural_networks.py"
)
neural_networks = importlib.util.module_from_spec(spec)
spec.loader.exec_module(neural_networks)
SimpleNeuralNetwork = neural_networks.SimpleNeuralNetwork

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

# Set style for better plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## Step 1: Load and Explore Dataset

In [ ]:
# Load dataset from CSV
print("Loading student dataset...")
df = pd.read_csv('data/neural_networks/student_degree_dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

print(f"\nDataset Statistics:")
print(df.describe())

print(f"\nDegree Category Distribution:")
category_counts = df['degree_category'].value_counts().sort_index()
print(category_counts)

# Visualize class distribution
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
category_counts.plot(kind='bar', color='steelblue')
plt.title('Class Distribution')
plt.xlabel('Degree Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
category_counts.plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title('Class Distribution (Percentage)')
plt.ylabel('')
plt.tight_layout()
plt.show()

print(f"\nClass Imbalance Ratio: {category_counts.max() / category_counts.min():.2f}x")

## Step 2: Advanced Data Preprocessing

In [ ]:
# Prepare features and labels
feature_columns = ['attendance', 'quiz_avg', 'assignment_avg', 'midterm_score',
                   'project_score', 'study_hours_per_week', 'participation_score']
X = df[feature_columns].values
y_categories = df['degree_category'].values

# Map categories to numbers
category_mapping = {
    'Bad': 0,
    'Acceptable': 1,
    'Good': 2,
    'Very Good': 3,
    'Excellent': 4
}
y = np.array([category_mapping[cat] for cat in y_categories])

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Unique classes: {np.unique(y)}")
print(f"Class distribution: {np.bincount(y)}")

# One-hot encode
def one_hot_encode(y, num_classes=5):
    encoded = np.zeros((len(y), num_classes))
    encoded[np.arange(len(y)), y] = 1
    return encoded

y_encoded = one_hot_encode(y)
print(f"One-hot encoded shape: {y_encoded.shape}")

In [ ]:
# Feature Engineering: Add polynomial features (degree 2)
print("Creating polynomial features...")
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
X_poly = poly.fit_transform(X)
print(f"Original features: {X.shape[1]}")
print(f"Polynomial features: {X_poly.shape[1]}")

# Use polynomial features
X_enhanced = X_poly

# Split data
X_train, X_test, y_train, y_test, y_train_labels, y_test_labels = train_test_split(
    X_enhanced, y_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")
print(f"Classes: {y_encoded.shape[1]}")

In [ ]:
# Advanced Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled using StandardScaler")
print(f"Training mean: {X_train_scaled.mean(axis=0)[:5]}")
print(f"Training std: {X_train_scaled.std(axis=0)[:5]}")

# Class balancing using class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)
class_weight_dict = dict(enumerate(class_weights))
print(f"\nClass weights for balancing: {class_weight_dict}")

# Visualize feature distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, col in enumerate(feature_columns):
    if idx < len(axes):
        axes[idx].hist(df[col], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Frequency')
        axes[idx].set_title(f'{col} Distribution')
        axes[idx].grid(True, alpha=0.3)

# Hide unused subplot
axes[7].axis('off')
plt.tight_layout()
plt.show()

## Step 3: Class Balancing with Oversampling

In [ ]:
# Simple oversampling for minority classes
def oversample_minority_classes(X, y, y_labels, target_samples=400):
    """Oversample minority classes to balance the dataset"""
    unique_classes, class_counts = np.unique(y_labels, return_counts=True)
    
    X_balanced = [X]
    y_balanced = [y]
    y_labels_balanced = [y_labels]
    
    for class_idx, count in zip(unique_classes, class_counts):
        if count < target_samples:
            # Find samples of this class
            class_mask = y_labels == class_idx
            X_class = X[class_mask]
            y_class = y[class_mask]
            y_labels_class = y_labels[class_mask]
            
            # Calculate how many samples to add
            n_samples_needed = target_samples - count
            
            # Random oversampling with replacement
            indices = np.random.choice(len(X_class), size=n_samples_needed, replace=True)
            X_oversampled = X_class[indices]
            y_oversampled = y_class[indices]
            y_labels_oversampled = y_labels_class[indices]
            
            X_balanced.append(X_oversampled)
            y_balanced.append(y_oversampled)
            y_labels_balanced.append(y_labels_oversampled)
    
    X_balanced = np.vstack(X_balanced)
    y_balanced = np.vstack(y_balanced)
    y_labels_balanced = np.hstack(y_labels_balanced)
    
    # Shuffle
    shuffle_idx = np.random.permutation(len(X_balanced))
    X_balanced = X_balanced[shuffle_idx]
    y_balanced = y_balanced[shuffle_idx]
    y_labels_balanced = y_labels_balanced[shuffle_idx]
    
    return X_balanced, y_balanced, y_labels_balanced

# Apply oversampling
print("Before oversampling:")
print(f"Class distribution: {np.bincount(y_train_labels)}")

X_train_balanced, y_train_balanced, y_train_labels_balanced = oversample_minority_classes(
    X_train_scaled, y_train, y_train_labels, target_samples=300
)

print("\nAfter oversampling:")
print(f"Class distribution: {np.bincount(y_train_labels_balanced)}")
print(f"Training samples: {len(X_train_balanced)}")

In [ ]:
class AdvancedNeuralNetwork(SimpleNeuralNetwork):
    """Extended neural network with learning rate scheduling and early stopping"""
    
    def __init__(self, layers, learning_rate=0.01, lr_decay=0.95, min_lr=0.0001):
        super().__init__(layers, learning_rate)
        self.initial_lr = learning_rate
        self.lr_decay = lr_decay
        self.min_lr = min_lr
        self.current_lr = learning_rate
        self.best_loss = float('inf')
        self.patience = 20
        self.wait = 0
        
    def update_learning_rate(self, epoch):
        """Decay learning rate"""
        self.current_lr = max(self.initial_lr * (self.lr_decay ** epoch), self.min_lr)
        self.learning_rate = self.current_lr
    
    def train_with_early_stopping(self, X_train, y_train, X_val, y_val, 
                                   epochs=500, verbose=True):
        """Train with early stopping and learning rate scheduling"""
        loss_history = []
        val_loss_history = []
        best_weights = None
        best_biases = None
        
        for epoch in range(epochs):
            # Update learning rate
            self.update_learning_rate(epoch)
            
            # Forward pass
            activations, z_values = self.forward(X_train)
            
            # Calculate training loss
            train_loss = np.mean((activations[-1] - y_train) ** 2)
            loss_history.append(train_loss)
            
            # Validation loss
            val_activations, _ = self.forward(X_val)
            val_loss = np.mean((val_activations[-1] - y_val) ** 2)
            val_loss_history.append(val_loss)
            
            # Early stopping
            if val_loss < self.best_loss:
                self.best_loss = val_loss
                self.wait = 0
                # Save best model
                best_weights = [w.copy() for w in self.weights]
                best_biases = [b.copy() for b in self.biases]
            else:
                self.wait += 1
                if self.wait >= self.patience:
                    if verbose:
                        print(f"\nEarly stopping at epoch {epoch + 1}")
                    break
            
            # Backward pass
            weight_gradients, bias_gradients = self.backward(X_train, y_train, activations, z_values)
            
            # Update weights
            self.update_weights(weight_gradients, bias_gradients)
            
            if verbose and (epoch + 1) % 50 == 0:
                print(f"Epoch {epoch + 1}/{epochs}, LR: {self.current_lr:.6f}, "
                      f"Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        
        # Restore best weights
        if best_weights is not None:
            self.weights = best_weights
            self.biases = best_biases
        
        return loss_history, val_loss_history

# Create validation set
X_train_final, X_val, y_train_final, y_val, y_train_final_labels, y_val_labels = train_test_split(
    X_train_balanced, y_train_balanced, y_train_labels_balanced, 
    test_size=0.15, random_state=42, stratify=y_train_labels_balanced
)

print(f"Final training set: {X_train_final.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test_scaled.shape[0]} samples")

# Create advanced neural network: Deeper architecture
# Input features (polynomial) -> 64 -> 128 -> 64 -> 32 -> 5 output
input_size = X_train_final.shape[1]
print(f"\nCreating advanced neural network...")
print(f"Architecture: {input_size} -> 64 -> 128 -> 64 -> 32 -> 5")

nn_advanced = AdvancedNeuralNetwork(
    layers=[input_size, 64, 128, 64, 32, 5],
    learning_rate=0.01,
    lr_decay=0.98,
    min_lr=0.0001
)

In [ ]:
print("Training advanced neural network...")
print("=" * 60)

train_loss_history, val_loss_history = nn_advanced.train_with_early_stopping(
    X_train_final, y_train_final, X_val, y_val,
    epochs=500,
    verbose=True
)

print("\n" + "=" * 60)
print("Training complete!")
print(f"Final training loss: {train_loss_history[-1]:.6f}")
print(f"Final validation loss: {val_loss_history[-1]:.6f}")
print(f"Best validation loss: {min(val_loss_history):.6f}")

# Plot training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_loss_history, label='Training Loss', linewidth=2)
plt.plot(val_loss_history, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_loss_history[-100:], label='Training Loss (last 100)', linewidth=2)
plt.plot(val_loss_history[-100:], label='Validation Loss (last 100)', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss (Last 100 Epochs)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Evaluate Model Performance

In [ ]:
# Make predictions on test set
print("Evaluating on test set...")
predictions = nn_advanced.predict(X_test_scaled)
predicted_classes = np.argmax(predictions, axis=1)
actual_classes = np.argmax(y_test, axis=1)

# Calculate metrics
accuracy = accuracy_score(actual_classes, predicted_classes)
f1_macro = f1_score(actual_classes, predicted_classes, average='macro')
f1_weighted = f1_score(actual_classes, predicted_classes, average='weighted')
precision = precision_score(actual_classes, predicted_classes, average='weighted')
recall = recall_score(actual_classes, predicted_classes, average='weighted')

print("\n" + "=" * 60)
print("MODEL EVALUATION RESULTS")
print("=" * 60)
print(f"\nTest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")
print(f"Precision (Weighted): {precision:.4f}")
print(f"Recall (Weighted): {recall:.4f}")

# Classification report
category_names = ['Bad', 'Acceptable', 'Good', 'Very Good', 'Excellent']
print("\n" + "=" * 60)
print("Detailed Classification Report:")
print("=" * 60)
print(classification_report(actual_classes, predicted_classes, 
                          target_names=category_names, 
                          labels=[0, 1, 2, 3, 4],
                          zero_division=0))

# Confusion matrix
cm = confusion_matrix(actual_classes, predicted_classes)
print("\n" + "=" * 60)
print("Confusion Matrix:")
print("=" * 60)
print(cm)

## Step 7: Visualize Results

In [ ]:
# Comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Confusion Matrix
im = axes[0, 0].imshow(cm, cmap='Blues', interpolation='nearest')
axes[0, 0].set_xticks(range(5))
axes[0, 0].set_yticks(range(5))
axes[0, 0].set_xticklabels(category_names, rotation=45, ha='right')
axes[0, 0].set_yticklabels(category_names)
axes[0, 0].set_xlabel('Predicted', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Actual', fontsize=12, fontweight='bold')
axes[0, 0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
for i in range(5):
    for j in range(5):
        axes[0, 0].text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=11, fontweight='bold',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[0, 0])

# 2. Normalized Confusion Matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
im2 = axes[0, 1].imshow(cm_normalized, cmap='Greens', interpolation='nearest', vmin=0, vmax=1)
axes[0, 1].set_xticks(range(5))
axes[0, 1].set_yticks(range(5))
axes[0, 1].set_xticklabels(category_names, rotation=45, ha='right')
axes[0, 1].set_yticklabels(category_names)
axes[0, 1].set_xlabel('Predicted', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Actual', fontsize=12, fontweight='bold')
axes[0, 1].set_title('Normalized Confusion Matrix', fontsize=14, fontweight='bold')
for i in range(5):
    for j in range(5):
        axes[0, 1].text(j, i, f'{cm_normalized[i, j]:.2f}', ha='center', va='center',
                    fontsize=10, fontweight='bold',
                    color='white' if cm_normalized[i, j] > 0.5 else 'black')
plt.colorbar(im2, ax=axes[0, 1])

# 3. Per-class Accuracy
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
axes[1, 0].bar(category_names, per_class_accuracy, color='steelblue', edgecolor='black')
axes[1, 0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[1, 0].set_title('Per-Class Accuracy', fontsize=14, fontweight='bold')
axes[1, 0].set_ylim([0, 1])
axes[1, 0].grid(True, alpha=0.3, axis='y')
axes[1, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(per_class_accuracy):
    axes[1, 0].text(i, v + 0.02, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')

# 4. Metrics Comparison
metrics = ['Accuracy', 'F1 (Macro)', 'F1 (Weighted)', 'Precision', 'Recall']
values = [accuracy, f1_macro, f1_weighted, precision, recall]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
axes[1, 1].barh(metrics, values, color=colors, edgecolor='black')
axes[1, 1].set_xlabel('Score', fontsize=12, fontweight='bold')
axes[1, 1].set_title('Model Metrics', fontsize=14, fontweight='bold')
axes[1, 1].set_xlim([0, 1])
axes[1, 1].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(values):
    axes[1, 1].text(v + 0.01, i, f'{v:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print(f"OVERALL MODEL PERFORMANCE: {accuracy*100:.2f}% Accuracy")
print(f"{'='*60}")

In [ ]:
# Analyze feature importance by examining weights
# For the first layer, larger absolute weights indicate more important features
first_layer_weights = nn_advanced.weights[0]
feature_importance = np.abs(first_layer_weights).mean(axis=1)

# Get original feature names (first 7 are original, rest are polynomial)
original_feature_names = feature_columns
if len(feature_importance) > len(original_feature_names):
    # For polynomial features, show top important ones
    top_indices = np.argsort(feature_importance)[-15:][::-1]
    top_importances = feature_importance[top_indices]
    
    # Create feature names for polynomial features
    poly_feature_names = poly.get_feature_names_out(feature_columns)
    top_feature_names = [poly_feature_names[i] for i in top_indices]
else:
    top_feature_names = original_feature_names
    top_importances = feature_importance

# Visualize feature importance
plt.figure(figsize=(12, 8))
plt.barh(range(len(top_feature_names)), top_importances, color='steelblue', edgecolor='black')
plt.yticks(range(len(top_feature_names)), top_feature_names)
plt.xlabel('Average Absolute Weight', fontsize=12, fontweight='bold')
plt.title('Top Feature Importance (First Layer Weights)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("Top 10 Most Important Features:")
for i, (name, imp) in enumerate(zip(top_feature_names[:10], top_importances[:10]), 1):
    print(f"{i:2d}. {name:30s}: {imp:.4f}")

## Step 9: Test on Sample Students

In [ ]:
# Test on sample students
test_students = [
    {
        'name': 'Excellent Student',
        'features': [95.0, 90.0, 92.0, 88.0, 95.0, 35.0, 95.0]
    },
    {
        'name': 'Very Good Student',
        'features': [88.0, 82.0, 85.0, 80.0, 88.0, 28.0, 85.0]
    },
    {
        'name': 'Good Student',
        'features': [80.0, 75.0, 78.0, 72.0, 80.0, 20.0, 75.0]
    },
    {
        'name': 'Acceptable Student',
        'features': [70.0, 60.0, 65.0, 58.0, 62.0, 15.0, 60.0]
    },
    {
        'name': 'Struggling Student',
        'features': [55.0, 45.0, 50.0, 48.0, 52.0, 10.0, 50.0]
    }
]

print("=" * 70)
print("PREDICTIONS ON SAMPLE STUDENTS")
print("=" * 70)

for student in test_students:
    # Create polynomial features
    features = np.array([student['features']])
    features_poly = poly.transform(features)
    features_scaled = scaler.transform(features_poly)
    
    # Make prediction
    prediction = nn_advanced.predict(features_scaled)
    predicted_class_idx = np.argmax(prediction)
    predicted_category = category_names[predicted_class_idx]
    confidence = prediction[0][predicted_class_idx]
    
    print(f"\n{student['name']}:")
    print(f"  Features: {student['features']}")
    print(f"  Predicted: {predicted_category} (confidence: {confidence:.4f} = {confidence*100:.2f}%)")
    print(f"  Class Probabilities:")
    for i, cat in enumerate(category_names):
        prob = prediction[0][i]
        bar = '█' * int(prob * 20)
        print(f"    {cat:15s}: {prob:.4f} ({prob*100:5.2f}%) {bar}")

print("\n" + "=" * 70)

## Step 10: Comparison with Project 7

Let's compare the performance improvements:

In [ ]:
# Load Project 7 model info if available
try:
    import json
    with open('data/neural_networks/student_model_info.json', 'r') as f:
        project7_info = json.load(f)
    project7_accuracy = project7_info.get('accuracy', 0.62)
except:
    project7_accuracy = 0.62  # Default from previous runs

# Comparison
comparison_data = {
    'Metric': ['Architecture', 'Features', 'Class Balancing', 
               'Learning Rate', 'Epochs', 'Accuracy'],
    'Project 7': ['7→32→16→5', '7 (original)', 'No', 
                  'Fixed 0.01', '200', f'{project7_accuracy*100:.2f}%'],
    'Project 8': ['Poly→64→128→64→32→5', '28 (polynomial)', 'Yes (oversampling)', 
                  'Adaptive (0.01→0.0001)', '500 (early stop)', f'{accuracy*100:.2f}%']
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "=" * 70)
print("PROJECT 7 vs PROJECT 8 COMPARISON")
print("=" * 70)
print(comparison_df.to_string(index=False))

improvement = ((accuracy - project7_accuracy) / project7_accuracy) * 100
print(f"\n{'='*70}")
print(f"Accuracy Improvement: {improvement:+.2f}%")
print(f"Absolute Improvement: {(accuracy - project7_accuracy)*100:+.2f} percentage points")
print(f"{'='*70}")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
axes[0].bar(['Project 7', 'Project 8'], 
            [project7_accuracy, accuracy], 
            color=['lightcoral', 'steelblue'], 
            edgecolor='black', linewidth=2)
axes[0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[0].set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylim([0, 1])
axes[0].grid(True, alpha=0.3, axis='y')
for i, (label, val) in enumerate(zip(['Project 7', 'Project 8'], 
                                     [project7_accuracy, accuracy])):
    axes[0].text(i, val + 0.02, f'{val:.3f}', ha='center', va='bottom', 
                fontweight='bold', fontsize=11)

# Improvement visualization
axes[1].barh(['Accuracy Improvement'], [improvement], 
             color='green' if improvement > 0 else 'red', 
             edgecolor='black', linewidth=2)
axes[1].set_xlabel('Improvement (%)', fontsize=12, fontweight='bold')
axes[1].set_title('Performance Improvement', fontsize=14, fontweight='bold')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[1].grid(True, alpha=0.3, axis='x')
axes[1].text(improvement + (1 if improvement > 0 else -1), 0, 
            f'{improvement:+.2f}%', ha='left' if improvement > 0 else 'right', 
            va='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()